# A400300: Bijective Davis numbers

Bijective version of Davis numbers (A384537).

In [1]:
import numpy as np
from numba import njit
from math import isqrt
from time import perf_counter

#known Davis numbers (A384537)
OLD = [16,27,64,256,729,1024,3125,4096,4617,16384,19683,29767,65536,
       255987,262144,395847,531441,631463,823543,1048576,1332331,
       4194304,9765625,14348907,16777216,25640947,67108864]

@njit
def make_spf(limit):
    spf = np.zeros(limit // 2 + 1, np.uint16)
    for p in range(3, int(limit ** 0.5) + 1, 2):
        if spf[p // 2] == 0:
            for m in range(p * p, limit + 1, 2 * p):
                if spf[m // 2] == 0:
                    spf[m // 2] = p
    return spf

@njit
def factor(n, spf):
    result = []
    if n % 2 == 0:
        e = 0
        while n % 2 == 0:
            n //= 2
            e += 1
        result.append((2, e))
    while n > 1:
        p = int(spf[n // 2]) if spf[n // 2] else n
        e = 0
        while n % p == 0:
            n //= p
            e += 1
        result.append((p, e))
    return result

@njit
def value(tokens, b, n, bijective=True):
    val = 0
    for x in tokens:
        digs = []
        while x:
            if bijective:
                x, r = divmod(x - 1, b)
                digs.append(r + 1)
            else:
                x, r = divmod(x, b)
                digs.append(r)

        for k in range(len(digs) - 1, -1, -1):
            val = val * b + digs[k]
            if val > n:
                return val
    return val

@njit
def matches(n, spf, bijective=True):
    fs = factor(n, spf)
    if len(fs) == 1 and fs[0][1] == 1:
        return [0][:0]
    tokens = []
    for p, e in fs:
        tokens.append(p)
        if e > 1:
            tokens.append(e)
    hits = []
    if sum(tokens) == n:
        hits.append(1)
    last, largest = tokens[-1], max(tokens)
    # Every possible base <= largest must divide n - last.
    divs = [1]
    for p, e in factor(n - last, spf):
        size = len(divs)
        power = 1
        for _ in range(e):
            power *= p
            for i in range(size):
                d = divs[i] * power
                if d <= largest:
                    divs.append(d)
    for b in divs:
        if b >= 2 and value(tokens, b, n, bijective) == n:
            hits.append(b)
    # Larger bases: each token occupies exactly one digit.
    lo, hi = largest + 1, n
    while lo <= hi:
        b = (lo + hi) // 2
        val = 0
        for x in tokens:
            val = val * b + x
            if val > n:
                break
        if val == n:
            hits.append(b)
            break
        if val < n:
            lo = b + 1
        else:
            hi = b - 1
    hits.sort()
    return hits

@njit
def original_bases(n, tokens):
    lo, hi = max(2, max(tokens) + 1), n
    while lo <= hi:
        b = (lo + hi) // 2
        value = 0
        for x in tokens:
            value = value * b + x
            if value > n:
                break
        if value == n:
            return [b]
        if value < n:
            lo = b + 1
        else:
            hi = b - 1
    return []
@njit
def scan(start, limit, spf, old_mask):
    found = []
    for n in range(start, limit + 1):
        if old_mask[n] or n == 4 or (n % 2 and spf[n // 2] == 0) or (n % 2 == 0 and n % 4 != 0 and spf[n // 4] == 0):
            continue
        bs = matches(n, spf)
        if bs:
            found.append((n, bs))
    return found


def search(limit=67_108_864, start=4):
    if not 4 <= start <= limit:
        raise ValueError('Require 4 <= start <= limit.')
    tic = perf_counter()
    spf = make_spf(limit)
    old_mask = np.zeros(limit + 1, np.bool_)
    for n in OLD:
        if n <= limit:
            old_mask[n] = True
    print('[1] Verify original Davis numbers', flush=True)
    found = {}
    for n in OLD:
        if n <= limit:
            bs = list(matches(n, spf))
            if bs:
                print(f'{n}: KEEP, bases={bs}')
                found[n] = bs
            else:
                original = list(matches(n, spf, False))
                print(f'{n}: REJECT, original bases={original}')
    print('\n[2-0] Verify 4', flush=True)
    bs = list(matches(4, spf))
    print(f'4: bases={bs}')
    if start <= 4 and bs:
        found[4] = bs
    print('\n[2-1] Search for new values', flush=True)
    for n, bs in scan(start, limit, spf, old_mask):
        found[n] = list(bs)
        print(f'NEW {n}: bases={list(bs)}')
    print('\n[3] Complete bijective Davis sequence')
    print(sorted(found))
    print(f'Total: {len(found)}, elapsed: {perf_counter()-tic:.2f}s')
    return found

if __name__ == '__main__':
    search()


[1] Verify original Davis numbers
16: KEEP, bases=[6]
27: KEEP, bases=[8]
64: KEEP, bases=[29]
256: KEEP, bases=[124]
729: KEEP, bases=[241]
1024: KEEP, bases=[507]
3125: KEEP, bases=[624]
4096: KEEP, bases=[2042]
4617: KEEP, bases=[11]
16384: KEEP, bases=[8185]
19683: KEEP, bases=[6558]
29767: KEEP, bases=[12]
65536: KEEP, bases=[32760]
255987: REJECT, original bases=[2]
262144: KEEP, bases=[131063]
395847: KEEP, bases=[362]
531441: KEEP, bases=[177143]
631463: KEEP, bases=[300]
823543: KEEP, bases=[117648]
1048576: KEEP, bases=[524278]
1332331: KEEP, bases=[57]
4194304: KEEP, bases=[2097141]
9765625: KEEP, bases=[1953123]
14348907: KEEP, bases=[4782964]
16777216: KEEP, bases=[8388596]
25640947: KEEP, bases=[1228]
67108864: KEEP, bases=[33554419]

[2-0] Verify 4
4: bases=[1]

[2-1] Search for new values
NEW 416: bases=[13]
NEW 1155: bases=[2]
NEW 7371: bases=[13]
NEW 17931: bases=[2]
NEW 19197: bases=[79]
NEW 27968: bases=[23]
NEW 128512: bases=[251]
NEW 671232: bases=[23]
NEW 1543580